# Notebook 3: Dependency Parsing

In this booklet, we will go through the dependency parsing python module, which mainly houses code that implements the CG3 -> CoNLL-U post-processing step. The main CG3 call is done through the `disambiguation.py` module, but the `dependency.py` module adds a function that can directly disambiguate and also parse dependencies of natural Ojibwe text. 

### Running the dependency grammar (and also the disambiguation )

To successfully parse dependencies, we first need the text to be disambiguated. This is because we need to have as precise morphological information as possible to accurately parse dependencies, since the links made between words are dependent on various information such as their POS, their number/obviation (in case of nouns), and their encoded arguments (in case of verbs).  For this reason, the main entry point of the dependency module (`parse_dependencies()`) first runs the disambiguation grammar over the input text, then runs the dependency module to parse syntactic dependencies. 

To illustrate this, we will work with the following Ojibwe sentence, the same one used in the "Full example (Ojibwe)" section of [the overview](../01_overview.md).

**Ogii-piisibidoon 'i mazina'igan.** ([OPD entry](https://ojibwe.lib.umn.edu/main-entry/biisibidoon-vti2))

Translation: He shredded the document.	

In [23]:
# this block can be ignored, just setting up the path
import sys
import os
sys.path.append(os.path.abspath("../../src"))  

assert(any(list((map(lambda x: os.path.isfile(x + '/grammar_modules/dependency.py'), sys.path)))))


In [ ]:
from pathlib import Path
from src.grammar_modules.fst import Fst
from src.grammar_modules.dependency import parse_dependencies, cg3_to_conllu_block
REPO_ROOT = Path(os.getcwd()).resolve().parents[1]

print(REPO_ROOT / "data" / "fst" / "ojibwe.fomabin")
# getting the FST (make sure path is correct)
fst = Fst(str(REPO_ROOT / "data" / "fst" / "ojibwe.fomabin"))
# path to the disambiguation grammar
disamb_grammar = str(REPO_ROOT / "data" / "grammars" / "disambiguation.cg3")
# path to the dependency grammar
dep_grammar = str(REPO_ROOT / "data" / "grammars" / "dependency.cg3")
# sentence to be analyzed
sent = "Ogii-piisibidoon 'i mazina'igan."

# calling parse_dependencies() with verbose = True to see output
deps = parse_dependencies(sent, dep_grammar, disamb_grammar, fst, verbose=True)

Before parsing (disambiguated text):
"<ogii-piisibidoon>"
	"biisibidoon" PVTense/gii VTI Ind Pos Neu 3SgProxSubj 0SgObj
"<'i>"
	"'i" PRONDem NI Sg
"<mazina'igan>"
	"mazina'igan" NI Sg
"<.>"


--------------------
After parsing:
"<ogii-piisibidoon>"
	"biisibidoon" PVTense/gii VTI Ind Pos Neu 3SgProxSubj 0SgObj VERB #1->1
"<'i>"
	"'i" PRONDem NI Sg @det DET #2->3
"<mazina'igan>"
	"mazina'igan" NI Sg @obj NOUN #3->1
"<.>"




### Converting to ConLL-U

Now that we have the CG3 output for the sentence, we can use the `cg3_to_conllu_block()` function to convert this into CoNLL-U format. If curious, the logic for the conversion can be found directly in the code. At a high level, we are simply taking the CG3 outputs for each word, and mapping them to their corresponding CoNLL-U column.


In [21]:
# convert to conllu format
# for individual sentences the sent_id (second argument) is not important, this argument is mainly used when building the corpus
conllu_block = cg3_to_conllu_block(deps, 1)
print(conllu_block)

# sent_id = 1
# text = ogii-piisibidoon 'i mazina'igan .
1	ogii-piisibidoon	biisibidoon	VERB	PVTense/gii|VTI|Ind|Pos|Neu|3SgProxSubj|0SgObj|VERB	_	0	root	_	_
2	'i	'i	DET	PRONDem|NI|Sg|DET	_	3	det	_	_
3	mazina'igan	mazina'igan	NOUN	NI|Sg|NOUN	_	1	obj	_	_
4	.	.	PUNCT	.	_	1	punct	_	_




### Visualizing the CoNLL-U

Finally, we will steal a function from the `corpus.py` module to quickly visualize our CoNLL-U output. To do this, will create a temp file with the CoNLL-U block that we parsed above.

In [22]:
import tempfile
from pathlib import Path
import os
from src.corpus import visualise_conllu

# create a temp .txt file
tmp = tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".txt", delete=False)
try:
    tmp.write(conllu_block)
    tmp.flush()
    corpus_path = Path(tmp.name)  # <- pass this path
finally:
    tmp.close()

# visualise sentence
visualise_conllu(corpus_path, 1)

# delete the 
os.unlink(corpus_path)


visualising sentence 1 from tmp0j0eqry_.txt

As we can see above, two dependencies were parsed. First, *'i*, a demonstrative pronoun (according to the FST) was linked to the following noun *mazina'igan*, since they are agreeing in animacy and number (both `NISg`). Then, the noun was linked to the verb as an object, because the verb had an inanimate singular object (`0SgObj`) agreement slot, which matched the noun for both animacy and number. 

### Summary of the `dpendency.py` module

As an important note, the dependency we went through was relatively simple, where every element was able to be disambiguated fully by the disambiguation grammar, and consequently every element was able to be linked to another element (or act as a root in the case of the verb). To play around more with the model, the easiest way would be to look at the OPD for interesting example sentences, and paste them into this pipeline to see what is and what is not successfully parsed.

That's it for the dependency module!